# preprocess SMYLE data

In [ ]:
%load_ext autoreload
%autoreload 2
import xarray as xr 
import numpy as np  
import cftime
import copy
import scipy.stats
from scipy import signal
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
%matplotlib inline

from SMYLEutils import calendar_utils as cal
from SMYLEutils import stat_utils as stat
from SMYLEutils import mapplot_utils as maps
from SMYLEutils import colorbar_utils as cbars
from SMYLEutils import io_utils as io

import dask
from dask.distributed import wait
dask.__version__

In [ ]:
from dask_jobqueue import PBSCluster
from dask.distributed import Client
#put your projection number here
proj = 'P93300070'
# Setup your PBSCluster
cluster = PBSCluster(
    cores=1, # The number of cores you want
    memory='150GB', # Amount of memory
    processes=15, # How many processes
    queue='casper', # The type of queue to utilize (/glade/u/apps/dav/opt/usr/bin/execcasper)
    log_directory= '/glade/derecho/scratch/smogen/dask/casper/logs',
    # local_directory= '/glade/derecho/scratch/${USER}/dask/derecho/local-dir',
    local_directory='$TMPDIR', # Use your local directory
    resource_spec='select=1:ncpus=3:mem=100GB', # Specify resources
    project=proj, # Input your project ID here
    walltime='02:00:00', # Amount of wall time
    interface='ext', # Interface to use
)
cluster.adapt(minimum=1,maximum=72,wait_count=60)
client=Client(cluster)

In [ ]:
client

## Define preprocessor

In [ ]:
def preprocessor(ds0,nlead,field):
    """ This preprocessor is applied on an individual timeseries file basis. Edit this appropriately
    for a your analysis to speed up processing. 
    """
    ds0 = cal.time_set_mid(ds0,'time')
    
    # # select the depth of your choosing
    # d0 = ds0[field].sel(z_t=0,method='nearest').isel(time=slice(0, nlead))
    d0 = ds0[field].isel(time=slice(0, nlead))

    d0 = d0.assign_coords(L=("time", np.arange(d0.sizes["time"])+1))
    d0 = d0.swap_dims({"time": "L"})
    d0 = d0.to_dataset(name=field)
    d0 = d0.reset_coords(["time"])
    d0["time"] = d0.time.expand_dims("Y")
    return d0   

In [ ]:
depth = 'surface'
time = 'monthly' 
timestep = 'month_1' # day_1, month_1
daily = False
init = 8 # 2, 5, 8, 11

In [ ]:
import xesmf as xe
from platform import python_version
print(xe.__version__)
obs = xr.open_dataset('/glade/work/smogen/SMYLE-extremes/OceanSODA-ETHZ_GRaCER_v2021a_1982-2020.nc')

## Load in data

In [ ]:
%%time
var = 'photoC_diat_zint_100m' # sp, diat, diaz
# SMYLE-Feb CO3 data
# process all 20 ensemble members, all start dates from 1970-2018:
field = var
datadir = '/glade/campaign/cesm/development/espwg/SMYLE/archive/'
casename = 'b.e21.BSMYLE.f09_g17.????-MM.EEE'
filetype = '.pop.h.'
filetemplate = datadir+casename+'/ocn/proc/tseries/month_1/'+casename+filetype+field+'.*.nc'
ens = 20 
nlead = 24
firstyear = 1970
lastyear  = 2019
startmonth = init

chunk = {}
smyle_diat = io.get_monthly_data(filetemplate,filetype,ens,nlead,field,firstyear,lastyear,startmonth,preprocessor,chunks=chunk)
smyle_sum_time = smyle_diat.time.load()

smyle_diat_sum = (smyle_diat.isel(z_t = [0,19]))[var] # for already integrated values
smyle_diat_sum.nbytes/1e9 #GB

In [ ]:
%%time
smyle_diat_sum = smyle_diat_sum.persist()

In [ ]:
%%time
var = 'photoC_diaz_zint_100m' # sp, diat, diaz
# SMYLE-Feb CO3 data
# process all 20 ensemble members, all start dates from 1970-2018:
field = var
datadir = '/glade/campaign/cesm/development/espwg/SMYLE/archive/'
casename = 'b.e21.BSMYLE.f09_g17.????-MM.EEE'
filetype = '.pop.h.'
filetemplate = datadir+casename+'/ocn/proc/tseries/month_1/'+casename+filetype+field+'.*.nc'
ens = 20 
nlead = 24
firstyear = 1970
lastyear  = 2019
startmonth = init

chunk = {}
smyle_diaz = io.get_monthly_data(filetemplate,filetype,ens,nlead,field,firstyear,lastyear,startmonth,preprocessor,chunks=chunk)

smyle_diaz_sum = (smyle_diaz)[var]
smyle_diaz_sum.nbytes/1e9 #GB

In [ ]:
%%time
smyle_diaz_sum = smyle_diaz_sum.persist()

In [ ]:
%%time
var = 'photoC_sp_zint_100m' # sp, diat, diaz
# SMYLE-Feb CO3 data
# process all 20 ensemble members, all start dates from 1970-2018:
field = var
datadir = '/glade/campaign/cesm/development/espwg/SMYLE/archive/'
casename = 'b.e21.BSMYLE.f09_g17.????-MM.EEE'
filetype = '.pop.h.'
filetemplate = datadir+casename+'/ocn/proc/tseries/month_1/'+casename+filetype+field+'.*.nc'
ens = 20 
nlead = 24
firstyear = 1970
lastyear  = 2019
startmonth = init

chunk = {}
smyle_sp = io.get_monthly_data(filetemplate,filetype,ens,nlead,field,firstyear,lastyear,startmonth,preprocessor,chunks=chunk)
smyle_sp_sum = (smyle_sp)[var]
smyle_sp_sum.nbytes/1e9 #GB

In [ ]:
%%time
smyle_sp_sum = smyle_sp_sum.persist()

In [ ]:
%%time
smyle_sum = smyle_diat_sum + smyle_diaz_sum + smyle_sp_sum

In [ ]:
%%time
smyle_sum = smyle_sum.load()

In [ ]:
%%time
regridder_smyle = xe.Regridder(smyle_sum.drop(['ULAT','ULONG']), obs, 'bilinear', periodic=True)
smyle_sum_rg = regridder_smyle(smyle_sum.drop(['ULAT','ULONG']))

In [ ]:
%%time
smyle_sum_rg.to_netcdf('/glade/derecho/scratch/smogen/Chl_extremes/monthly/SMYLE/'+ new_var +'.' + time + '.' + depth + '.' + str(init) + '.regrid.nc')
smyle_sum_time.to_netcdf('/glade/derecho/scratch/smogen/Chl_extremes/monthly/SMYLE/'+ new_var +'.' + time + '.' + str(init) + '.time.nc')

In [ ]:
del smyle_diat_sum, smyle_diaz_sum, smyle_sp_sum

In [ ]:
client.close()
client.shutdown()